# Algothon 2026 — Working Notebook

A plain scaffold to develop and test `getMyPosition`. Edit freely.

Layout:
1. Setup & load data
2. The strategy (edit this)
3. Local backtest (same accounting as `eval.py`)
4. Run it
5. Scratch space for your own experiments

Put `prices.txt` next to this notebook (or fix the path in the first cell).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


PRICES_FILE = "500prices.txt"          # <-- fix path if needed
prices_750 = pd.read_csv("750prices.txt", sep=r"\s+")  # (days, 51)

prices = pd.read_csv(PRICES_FILE, sep=r"\s+")   # (days, 51)
names  = prices.columns.tolist()
P      = prices.values.T                          # (51, days) — same shape eval.py uses
nInst, nDays = P.shape
print(f"{nInst} instruments x {nDays} days")
print("first few tickers:", names[:6])

## 2. The strategy — edit this cell

`getMyPosition(prcSoFar)` receives prices up to today, shape `(51, t)`, and returns
51 integer share positions. This is the residual mean-reversion strategy: remove
the top-K market themes with PCA, bet against each asset's recent private wiggle.

In [ ]:
FIT   = 130          # trailing days to fit the factor model
K     = 5            # number of market themes (PCA factors) to remove
W     = 20           # look-back window for the reversion signal
GROSS = 1_500_000    # dollar sizing knob (each name clips to $10k)

def getMyPosition(prcSoFar):
    n, t = prcSoFar.shape
    if t < FIT + 2:
        return np.zeros(n, dtype=int)

    # 1) log returns, last FIT days, drop ALGO (col 0 is the market index)
    R = np.log(prcSoFar[:, 1:] / prcSoFar[:, :-1]).T[-FIT:, 1:]   # (FIT, 50)

    # 2) centre, then PCA via SVD; Vt rows are the theme directions
    Zc = R - R.mean(axis=0)
    U, S, Vt = np.linalg.svd(Zc, full_matrices=False)

    # 3) rebuild from top-K themes, subtract -> private residual
    themes = (Zc @ Vt[:K].T) @ Vt[:K]
    resid  = Zc - themes

    # 4) bet against recent residual, risk-parity weighted
    sig = -resid[-W:].mean(axis=0) / (resid.std(axis=0) + 1e-9)

    # 5) winsorize, normalise to a dollar book, convert to shares
    s   = sig.std() + 1e-9
    sig = np.clip(sig, -3 * s, 3 * s)
    sig = sig / (np.abs(sig).sum() + 1e-9)
    px  = prcSoFar[:, -1]
    pos = np.zeros(n)
    pos[1:] = GROSS * sig / px[1:]
    return pos.astype(int)

## 3. Local backtest

Mirrors `eval.py` exactly: per-instrument $ limits, commissions (ALGO cheaper),
daily clip, and the score = μ·SR²/(SR²+1).

In [ ]:
def run_backtest(getpos, prc=P, numTestDays=250, verbose=False):
    nInst, nt = prc.shape
    commRate = np.full(nInst, 1e-4); commRate[0] = 2e-5
    posLim   = np.full(nInst, 10_000.0); posLim[0] = 100_000.0

    cash = 0.0; cur = np.zeros(nInst); tot = 0.0; val = 0.0; comm = 0.0
    pls = []; start = nt - numTestDays
    for t in range(start, nt + 1):
        px = prc[:, :t][:, -1]
        if t < nt:
            npos = getpos(prc[:, :t])
            lim  = (posLim / px).astype(int)
            npos = np.clip(npos, -lim, lim).astype(int)
        else:
            npos = cur.copy()
        d = npos - cur
        cash -= px.dot(d) + comm
        dv = px * np.abs(d); comm = np.sum(dv * commRate); tot += dv.sum()
        cur = npos; pv = cur.dot(px); pl = cash + pv - val; val = cash + pv
        if t > start:
            pls.append(pl)
            if verbose: print(f"day {t}: PL {pl:8.1f}  value {val:10.1f}")

    pls = np.array(pls); mu, sd = pls.mean(), pls.std()
    sr = np.sqrt(250) * mu / sd if sd > 0 else 0.0
    score = mu * (sr**2 / (sr**2 + 1)) if (mu > 0 and sd > 1e-10) else mu
    return dict(score=score, mean_pl=mu, sharpe=sr, volume=tot)

## 4. Run it

In [ ]:
res = run_backtest(getMyPosition)
print(f"Score        : {res['score']:.1f}")
print(f"Mean daily PL: ${res['mean_pl']:.1f}")
print(f"Ann Sharpe   : {res['sharpe']:.2f}")
print(f"Total volume : ${res['volume']/1e6:.1f}M  (need > $25k)")

## 5. Scratch space

Ideas to try (change the cell above, re-run cell 4):
- sweep `K`, `W`, `GROSS` and record the score
- add the ensemble: loop `for K in (4,5,6): for W in (18,20,22):` and average `sig`
- walk-forward: `run_backtest(getMyPosition, prc=P[:, :375])` to test on unseen-ish days
- inspect today's positions: `getMyPosition(P[:, :400])`

In [ ]:
# example: quick sweep over the sizing knob
for g in [500_000, 1_000_000, 1_500_000, 2_000_000]:
    GROSS = g
    r = run_backtest(getMyPosition)
    print(f"GROSS ${g/1e6:.1f}M -> score {r['score']:6.1f}  Sharpe {r['sharpe']:.2f}")
GROSS = 1_500_000   # reset

In [ ]:
# your experiments here
